# IBMED-D-26-00017 — Visual availability, sensitivity and utility (plan v3)

**Research question:** does access to medical images translate into measurable visual reliance, and at what computational cost?

Version 3 of the plan (`ANALYSIS_PLAN.md`) applies three changes from co-author review:
1. strata defined from dataset annotations (text-only solvability is exploratory only);
2. model-specific neutralization, interpreted within each model only;
3. **image-disjoint VQA-RAD split** (Wu et al., 2025). The pilot run on the original split is kept as a **sensitivity analysis**.

**Before running:**
- Settings → **GPU T4 x2**, Internet **On**; Secret `HF_TOKEN` attached.
- Inputs: `ibmed-expe` (**new version**), `vqa-rad`, and, if the pilot session is gone, the pilot results `results_ibmed.zip` uploaded as a dataset (e.g. `ibmed-pilot`).

Then **Run All** (about 1–1.5 h). The output is `/kaggle/working/results_ibmed_v3.zip`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Copy the scripts (and the bundled VQA-RAD split) into the working directory
import glob, os, shutil
src = glob.glob('/kaggle/input/**/common.py', recursive=True)
assert src, 'Scripts not found: add the ibmed-expe dataset (Add Input).'
code_dir = os.path.dirname(src[0])
os.chdir('/kaggle/working')
for f in glob.glob(code_dir + '/*'):
    if os.path.isfile(f):
        shutil.copy(f, '.')
    elif os.path.isdir(f):
        shutil.copytree(f, os.path.basename(f), dirs_exist_ok=True)
assert os.path.exists('data_splits/vqa_rad_fixed/test_fixed_split.json'), 'Old version of ibmed-expe: update the dataset (New Version).'
print(sorted(os.listdir('.')))

In [ ]:
# 3. Dependencies + HuggingFace login
!pip install -q -U "transformers>=4.50" "accelerate>=0.30" "bitsandbytes>=0.43" "open_clip_torch>=2.23" timm "huggingface_hub>=0.23" datasets
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
print('HuggingFace account:', whoami()['name'])

In [ ]:
# 4. Environment and precision (T4: 4-bit weights, float32 compute)
!python 00_check_env.py

In [ ]:
# 5. Keep the PILOT run (original VQA-RAD split) for the sensitivity analysis
import zipfile
def pilot_ok():
    return os.path.exists('work_pilot/predictions/mllm_medgemma.csv') and os.path.exists('work_pilot/data/questions.csv')
# (a) same Kaggle session: the pilot is still in work/
if not pilot_ok() and os.path.exists('work/predictions/mllm_medgemma.csv'):
    for d in ('predictions', 'results'):
        if os.path.exists('work/' + d):
            shutil.copytree('work/' + d, 'work_pilot/' + d, dirs_exist_ok=True)
    os.makedirs('work_pilot/data', exist_ok=True)
    shutil.copy('work/data/questions.csv', 'work_pilot/data/questions.csv')
# (b) pilot results uploaded as a dataset (zip, or auto-extracted by Kaggle)
if not pilot_ok():
    for z in glob.glob('/kaggle/input/**/results_ibmed*.zip', recursive=True):
        zipfile.ZipFile(z).extractall('work_pilot')
if not pilot_ok():
    hits = glob.glob('/kaggle/input/**/predictions/mllm_medgemma.csv', recursive=True)
    if hits:
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), 'work_pilot', dirs_exist_ok=True)
print('Pilot kept for sensitivity analysis:', pilot_ok())
# Start the main run from a clean state (images are kept: they are re-validated by step 01)
for d in ('work/predictions', 'work/features', 'work/results'):
    shutil.rmtree(d, ignore_errors=True)

In [ ]:
# 6. Data: image-disjoint VQA-RAD split (default) + SLAKE; annotations; stratified swaps
!python 01_prepare_data.py

In [ ]:
# 7. Reuse the pilot's SLAKE MedGemma answers ONLY if SLAKE questions and swap pairs are identical
import pandas as pd
os.makedirs('work/predictions', exist_ok=True); os.makedirs('work/results', exist_ok=True)
reused = False
if pilot_ok():
    new_q = pd.read_csv('work/data/questions.csv', keep_default_na=False)
    old_q = pd.read_csv('work_pilot/data/questions.csv', keep_default_na=False)
    def key(q):  # compare images by content hash (file name), not by absolute path
        t = q[(q.dataset == 'slake') & (q.split == 'test')].copy()
        t['swap'] = t['swap_image_path'].map(os.path.basename)
        return t[['qid', 'image_hash', 'swap', 'question', 'label']].astype(str).sort_values('qid').reset_index(drop=True)
    a, b = key(new_q), key(old_q)
    if a.equals(b):
        p = pd.read_csv('work_pilot/predictions/mllm_medgemma.csv')
        p[p.dataset == 'slake'].to_csv('work/predictions/mllm_medgemma.csv', index=False)
        for f in glob.glob('work_pilot/results/compute_mllm_*.csv'):
            shutil.copy(f, 'work/results/')
        reused = True
print('SLAKE MedGemma answers reused:', reused, '(if False, step 9 recomputes them: +1 h)')

In [ ]:
# 8. Classical fusion (frozen BiomedCLIP), 3 models x 5 seeds x 4 conditions  (~15 min)
!python 02_classical.py

In [ ]:
# 9. MedGemma zero-shot (only the missing answers: new VQA-RAD test questions)  (~45-60 min)
#    If the session stops: re-run cell 3, then this cell -- it resumes.
!python 03_mllm.py --backend medgemma

In [ ]:
# 10. Main analysis (plan v3) + sensitivity analysis on the pilot (original VQA-RAD split)
!python 04_analysis.py
if pilot_ok():
    !EXPE_WORK=/kaggle/working/work_pilot python 04_analysis.py > work_pilot_analysis.log 2>&1 && echo 'sensitivity analysis: OK'

In [ ]:
# 11. Package everything (main results + sensitivity analysis on the original split)
!cd work && zip -qr /kaggle/working/results_ibmed_v3.zip results predictions env.json data/questions.csv
if pilot_ok():
    shutil.rmtree('_s', ignore_errors=True); os.makedirs('_s')
    shutil.copytree('work_pilot/results', '_s/sensitivity_original_split')
    !cd _s && zip -qr /kaggle/working/results_ibmed_v3.zip sensitivity_original_split
print('-> Download /kaggle/working/results_ibmed_v3.zip (Output tab)')

In [ ]:
# 12. Quick look
from IPython.display import Image, display
display(Image('work/results/fig_sensitivity_utility.png'))
display(Image('work/results/fig_deltas.png'))
print(pd.read_csv('work/results/properties.csv').round(3).to_string(index=False))
print(open('work/results/data_checks.json').read())